## Configurando el espacio de trabajo :D!

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Que los floats se impriman con 4 decimales :D!
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
np.set_printoptions(precision = 4, suppress = True)

In [2]:
# Los datos a trabajar son estos
    # X = Horas de estudio (vaariable predictoria)
    # Y = Nota (variable respuesta)
datos = pd.DataFrame({
    'Estudiante': range(1, 11),
    'Horas': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Nota': [3.2, 4.5, 5.0, 6.8, 7.0, 8.5, 9.0, 9.2, 9.8, 10.0]
})

In [3]:
# Extraemos los vectores como float para evitar división entera, además de fijar las constantes que trabajaremos durante toda la práctica

x = datos['Horas'].to_numpy(dtype = float)
y = datos['Nota'].to_numpy(dtype = float)

n = len(x)      # Numero de observaciones
p = 1           # Numero de predictores
alpha = 0.05    # nivel de significancia para las pruebas e intervalos

## Modelo lineal por MCO

In [4]:
# Fórmula cerrada paso por paso :D!
x_bar = x.mean()
y_bar = y.mean()

Sxx = np.sum((x - x_bar) ** 2)          # Variablilidad de X
Sxy = np.sum((x - x_bar) * (y - y_bar)) # Covarianza de X y Y

print(f"x_barra = {x_bar:.4f}")
print(f"y_barra = {y_bar:.4f}")
print(f"Sxx     = {Sxx:.4f}")
print(f"Sxy     = {Sxy:.4f}")

x_barra = 5.5000
y_barra = 7.3000
Sxx     = 82.5000
Sxy     = 63.7000


In [5]:
# Estimadores MCO para la fórmula cerrada!
beta1 = Sxy / Sxx               # pendiente
beta0 = y_bar - beta1 * x_bar   # intercepto

# Valores ajustados y residuos:
y_hat = beta0 + beta1 * x
residuos = y - y_hat

print(f"beta0_hat = {beta0:.4f}")
print(f"beta1_hat = {beta1:.4f}")
print(f"\nModelo estimado:  y_hat = {beta0:.4f} + {beta1:.4f} * x")

beta0_hat = 3.0533
beta1_hat = 0.7721

Modelo estimado:  y_hat = 3.0533 + 0.7721 * x


In [6]:
# Si ambos métodos coinciden, es que la fórmula cerrada quedó bien implementada 🤠
X = np.column_stack([np.ones(n), x])    # matriz de 1's
beta_matricial = np.linalg.inv(X.T @ X) @ X.T @ y

print("Fórmula cerrada :", np.array([beta0, beta1]))
print("Vía matricial   :", beta_matricial)
print("Coinciden.?     :", np.allclose([beta0, beta1], beta_matricial))

Fórmula cerrada : [3.0533 0.7721]
Vía matricial   : [3.0533 0.7721]
Coinciden.?     : True


In [7]:
# Tabla para ver de dónde saldrá el SSE
tabla_ajuste = datos.copy()
tabla_ajuste['y_hat']    = y_hat
tabla_ajuste['residuo']  = residuos
tabla_ajuste['residuo2'] = residuos ** 2

tabla_ajuste

,Estudiante,Horas,Nota,y_hat,residuo,residuo2
0,1,1,3.2000,3.8255,-0.6255,0.3912
1,2,2,4.5000,4.5976,-0.0976,0.0095
2,3,3,5.0000,5.3697,-0.3697,0.1367
3,4,4,6.8000,6.1418,0.6582,0.4332
4,5,5,7.0000,6.9139,0.0861,0.0074
5,6,6,8.5000,7.6861,0.8139,0.6625
6,7,7,9.0000,8.4582,0.5418,0.2936
7,8,8,9.2000,9.2303,-0.0303,0.0009
8,9,9,9.8000,10.0024,-0.2024,0.0410
9,10,10,10.0000,10.7745,-0.7745,0.5999


## Tabla ANOVA

In [8]:
# Suma de cuadrados
SST = np.sum((y - y_bar) ** 2)      # variabilidad total de Y
SSR = np.sum((y_hat - y_bar) ** 2)  # variabilidad explicada (por el modelo)
SSE = np.sum((y - y_hat) ** 2)      # variabilidad residual

print(f"SST = {SST:.4f}")
print(f"SSR = {SSR:.4f}")
print(f"SSE = {SSE:.4f}")

# Verificación de la identidad SST = SSR + SSE
print(f"\nVerificación: {SST:.4f} = {SSR:.4f} + {SSE:.4f} = {SSR + SSE:.4f}")
print("Se cumple SST = SSR + SSE..?", np.isclose(SST, SSR + SSE))

SST = 51.7600
SSR = 49.1841
SSE = 2.5759

Verificación: 51.7600 = 49.1841 + 2.5759 = 51.7600
Se cumple SST = SSR + SSE..? True


In [9]:
# Grados de libertad :D!!!
df_reg   = p            # 1 predictor estimado (beta1)
df_error = n - p - 1    # perdemos 1 df por y_barra y 1 por beta1
df_total = n - 1        # perdemos 1 df al estimar y_barra

print(f"df_Regresión = p         = {df_reg}")
print(f"df_Error     = n - p - 1 = {df_error}")
print(f"df_Total     = n - 1     = {df_total}")
print(f"\nRegla clave: {df_total} = {df_reg} + {df_error} ->",
      df_total == df_reg + df_error)

df_Regresión = p         = 1
df_Error     = n - p - 1 = 8
df_Total     = n - 1     = 9

Regla clave: 9 = 1 + 8 -> True


In [10]:
# Cuadrados medios
MSR = SSR / df_reg      # cuadrado medio de regresión
MSE = SSE / df_error    # cuadrado medio del error = estimación de sigma^2
F_stat = MSR / MSE      # estadístico F (micky herramienta que usaremos a futuro :D!)

print(f"MSR = SSR/p         = {MSR:.4f}")
print(f"MSE = SSE/(n-p-1)   = {MSE:.4f}")   # Estimación de sigma^2
print(f"sigma_hat           = {np.sqrt(MSE):.4f}")
print(f"F   = MSR/MSE       = {F_stat:.4f}")

MSR = SSR/p         = 49.1841
MSE = SSE/(n-p-1)   = 0.3220
sigma_hat           = 0.5674
F   = MSR/MSE       = 152.7529


In [11]:
# Armamos la tabla ANOVA 🤠
anova = pd.DataFrame({
    'Fuente': ['Regresión', 'Error', 'Total'],
    'SS':     [SSR, SSE, SST],
    'df':     [df_reg, df_error, df_total],
    'MS':     [MSR, MSE, np.nan],
    'F':      [F_stat, np.nan, np.nan]
}).set_index('Fuente')

anova

,SS,df,MS,F
Fuente,,,,
Regresión,49.1841,1,49.1841,152.7529
Error,2.5759,8,0.3220,NaN
Total,51.7600,9,NaN,NaN


In [12]:
# R^2 = proporción de la variabilidad de Y que el modelo explica
# En regresión simple es igual al cuadrado de la correlación de Pearson
R2 = SSR / SST
r_pearson = Sxy / np.sqrt(Sxx * np.sum((y - y_bar) ** 2))

print(f"R^2             = {R2:.4f}  ({R2*100:.2f}% de la variabilidad explicada)")
print(f"r de Pearson    = {r_pearson:.4f}")
print(f"r^2             = {r_pearson**2:.4f}  -> igual a R^2..?", np.isclose(R2, r_pearson**2))

R^2             = 0.9502  (95.02% de la variabilidad explicada)
r de Pearson    = 0.9748
r^2             = 0.9502  -> igual a R^2..? True


## Significancia del modelo (Prueba F!)

In [13]:
# Valor crítico de F: cuantil (1 - alpha) de la F con (df_reg, df_error)
# p-valor: probabilidad de ver un F igual o más extremo si H0 fuera cierta
F_critico = stats.f.ppf(1 - alpha, df_reg, df_error)
p_valor_F = stats.f.sf(F_stat, df_reg, df_error)   # sf = 1 - cdf, más preciso en la cola

print(f"F observado             = {F_stat:.4f}")
print(f"F crítico (0.05, 1, 8)  = {F_critico:.4f}")
print(f"p-valor                 = {p_valor_F:.3e}")

F observado             = 152.7529
F crítico (0.05, 1, 8)  = 5.3177
p-valor                 = 1.712e-06


In [14]:
# Aplicamos la regla de decisión por los dos caminos :D!!
rechazo_por_critico = F_stat > F_critico
rechazo_por_pvalor  = p_valor_F < alpha

print(f"¿F > F_crítico?   {F_stat:.4f} > {F_critico:.4f}  ->  {rechazo_por_critico}")
print(f"¿p-valor < alpha? {p_valor_F:.3e} < {alpha}  ->  {rechazo_por_pvalor}")

if rechazo_por_critico:
    print("\nRechazamos H0: el modelo es globalmente significativo.")
    print("La varianza explicada es mucho mayor que la no explicada.")
else:
    print("\nNo rechazamos H0: no hay evidencia de relación lineal.")

¿F > F_crítico?   152.7529 > 5.3177  ->  True
¿p-valor < alpha? 1.712e-06 < 0.05  ->  True

Rechazamos H0: el modelo es globalmente significativo.
La varianza explicada es mucho mayor que la no explicada.


# Significancia de los coeficiente (Prueba t)

In [15]:
# Errores  de los coeficientes
SE_beta1 = np.sqrt(MSE / Sxx)
SE_beta0 = np.sqrt(MSE * (1/n + x_bar**2 / Sxx))

print(f"SE(beta1_hat) = sqrt(MSE/Sxx)              = {SE_beta1:.4f}")
print(f"SE(beta0_hat) = sqrt(MSE(1/n + x_bar^2/Sxx)) = {SE_beta0:.4f}")

SE(beta1_hat) = sqrt(MSE/Sxx)              = 0.0625
SE(beta0_hat) = sqrt(MSE(1/n + x_bar^2/Sxx)) = 0.3876


In [16]:
# Estadísticos t, valor crítico bilateral y p-valores
gl = n - 2

t_beta1 = beta1 / SE_beta1
t_beta0 = beta0 / SE_beta0

t_critico = stats.t.ppf(1 - alpha/2, gl)
p_valor_t1 = 2 * stats.t.sf(abs(t_beta1), gl)
p_valor_t0 = 2 * stats.t.sf(abs(t_beta0), gl)

print(f"t crítico (0.025, {gl}) = {t_critico:.4f}\n")
print(f"beta1: t = {t_beta1:.4f}  |t| > t_c -> {abs(t_beta1) > t_critico}  p = {p_valor_t1:.3e}")
print(f"beta0: t = {t_beta0:.4f}  |t| > t_c -> {abs(t_beta0) > t_critico}  p = {p_valor_t0:.3e}")

t crítico (0.025, 8) = 2.3060

beta1: t = 12.3593  |t| > t_c -> True  p = 1.712e-06
beta0: t = 7.8769  |t| > t_c -> True  p = 4.882e-05


In [17]:
# Tabla resumen de coeficientes (con estilo coqueto 🤠)
tabla_coef = pd.DataFrame({
    'Coeficiente': ['beta0 (intercepto)', 'beta1 (pendiente)'],
    'Estimación':  [beta0, beta1],
    'Error Est.':  [SE_beta0, SE_beta1],
    't':           [t_beta0, t_beta1],
    'p-valor':     [p_valor_t0, p_valor_t1],
    'Significativo': [abs(t_beta0) > t_critico, abs(t_beta1) > t_critico]
}).set_index('Coeficiente')

tabla_coef

,Estimación,Error Est.,t,p-valor,Significativo
Coeficiente,,,,,
beta0 (intercepto),3.0533,0.3876,7.8769,0.0000,True
beta1 (pendiente),0.7721,0.0625,12.3593,0.0000,True


## Verificación de la relación

In [18]:
# Comparación: F contra t^2
t1_cuadrado = t_beta1 ** 2
diferencia = abs(F_stat - t1_cuadrado)

print(f"F        = {F_stat:.6f}")
print(f"t_beta1² = ({t_beta1:.4f})² = {t1_cuadrado:.6f}")
print(f"\nDiferencia absoluta = {diferencia:.2e}")
print("¿F == t^2?:", np.isclose(F_stat, t1_cuadrado))

F        = 152.752906
t_beta1² = (12.3593)² = 152.752906

Diferencia absoluta = 0.00e+00
¿F == t^2?: True


In [19]:
# La equivalencia también se ve en los valores críticos y en los p-valores
print(f"F_crítico = {F_critico:.4f}   vs   t_crítico² = {t_critico**2:.4f}")
print("¿Coinciden?:", np.isclose(F_critico, t_critico**2))
print()
print(f"p-valor F = {p_valor_F:.6e}")
print(f"p-valor t = {p_valor_t1:.6e}")
print("Coinciden?", np.isclose(p_valor_F, p_valor_t1))

F_crítico = 5.3177   vs   t_crítico² = 5.3177
¿Coinciden?: True

p-valor F = 1.711655e-06
p-valor t = 1.711655e-06
Coinciden? True


In [20]:
# La equivalencia también se ve en los valores críticos y en los p-valores.
print(f"F_crítico = {F_critico:.4f}   vs   t_crítico^2 = {t_critico**2:.4f}")
print("¿Coinciden?:", np.isclose(F_critico, t_critico**2))
print()
print(f"p-valor F = {p_valor_F:.6e}")
print(f"p-valor t = {p_valor_t1:.6e}")
print("Coinciden?", np.isclose(p_valor_F, p_valor_t1))

F_crítico = 5.3177   vs   t_crítico^2 = 5.3177
¿Coinciden?: True

p-valor F = 1.711655e-06
p-valor t = 1.711655e-06
Coinciden? True


##  Intervalos de confianza para los coeficientes

In [21]:
# Margen de error = t_crítico * SE
me_beta0 = t_critico * SE_beta0
me_beta1 = t_critico * SE_beta1

IC_beta0 = (beta0 - me_beta0, beta0 + me_beta0)
IC_beta1 = (beta1 - me_beta1, beta1 + me_beta1)

print(f"IC 95% beta0: {beta0:.4f} +- {t_critico:.3f}*{SE_beta0:.4f} = "
      f"[{IC_beta0[0]:.4f}, {IC_beta0[1]:.4f}]")
print(f"IC 95% beta1: {beta1:.4f} +- {t_critico:.3f}*{SE_beta1:.4f} = "
      f"[{IC_beta1[0]:.4f}, {IC_beta1[1]:.4f}]")

IC 95% beta0: 3.0533 +- 2.306*0.3876 = [2.1594, 3.9472]
IC 95% beta1: 0.7721 +- 2.306*0.0625 = [0.6281, 0.9162]


In [22]:
# Chequeo del criterio "¿contiene al 0?" para cada coeficiente 🤠
contiene_cero_b0 = IC_beta0[0] <= 0 <= IC_beta0[1]
contiene_cero_b1 = IC_beta1[0] <= 0 <= IC_beta1[1]

tabla_IC = pd.DataFrame({
    'Coeficiente':   ['beta0', 'beta1'],
    'Estimación':    [beta0, beta1],
    'Límite Inf':    [IC_beta0[0], IC_beta1[0]],
    'Límite Sup':    [IC_beta0[1], IC_beta1[1]],
    '¿Contiene 0?':  [contiene_cero_b0, contiene_cero_b1],
    'Significativo': [not contiene_cero_b0, not contiene_cero_b1]
}).set_index('Coeficiente')

display(tabla_IC)

print(f"\nInterpretación de beta1: con 95% de confianza, cada hora adicional de estudio")
print(f"aumenta la nota entre {IC_beta1[0]:.3f} y {IC_beta1[1]:.3f} puntos.")
print("El intervalo NO contiene 0 -> rechazamos H0: beta1 = 0.")

,Estimación,Límite Inf,Límite Sup,¿Contiene 0?,Significativo
Coeficiente,,,,,
beta0,3.0533,2.1594,3.9472,False,True
beta1,0.7721,0.6281,0.9162,False,True



Interpretación de beta1: con 95% de confianza, cada hora adicional de estudio
aumenta la nota entre 0.628 y 0.916 puntos.
El intervalo NO contiene 0 -> rechazamos H0: beta1 = 0.


In [23]:
# Consistencia entre los tres enfoques
print("¿beta1 significativo según...")
print(f"  el IC (no contiene 0)?  {not contiene_cero_b1}")
print(f"  la prueba t (|t|>t_c)?  {abs(t_beta1) > t_critico}")
print(f"  la prueba F (F>F_c)?    {F_stat > F_critico}")

¿beta1 significativo según...
  el IC (no contiene 0)?  True
  la prueba t (|t|>t_c)?  True
  la prueba F (F>F_c)?    True


## Tabla de IC de la media e IP (intervalo de predicción)

In [24]:
def intervalos(x0, nivel=0.95):
    """Calcula, para un valor X0 dado:
       - y_hat0 : la predicción puntual
       - IC     : intervalo de confianza para E[Y|X0]  (el promedio)
       - IP     : intervalo de predicción para una Y individual
       Devuelve además ambos errores estándar para poder inspeccionarlos.
    """
    t_c = stats.t.ppf(1 - (1 - nivel)/2, n - 2)

    y_hat0 = beta0 + beta1 * x0  # predicción puntual
    leverage = 1/n + (x0 - x_bar)**2 / Sxx # término que crece al alejarse de x_barra

    SE_mean = np.sqrt(MSE * leverage)       # error de estimación
    SE_pred = np.sqrt(MSE * (1 + leverage))

    return {
        'y_hat':    y_hat0,
        'SE_mean':  SE_mean,
        'SE_pred':  SE_pred,
        'IC_inf':   y_hat0 - t_c * SE_mean,
        'IC_sup':   y_hat0 + t_c * SE_mean,
        'IP_inf':   y_hat0 - t_c * SE_pred,
        'IP_sup':   y_hat0 + t_c * SE_pred,
    }

In [25]:
# Aplicamos la función a las 10 horas observadas
filas = [intervalos(x0) for x0 in x]

tabla_intervalos = pd.DataFrame({
    'Horas':        x,
    'Nota':         y,
    'y_hat':        [f['y_hat'] for f in filas],
    'IC_inf':       [f['IC_inf'] for f in filas],
    'IC_sup':       [f['IC_sup'] for f in filas],
    'IP_inf':       [f['IP_inf'] for f in filas],
    'IP_sup':       [f['IP_sup'] for f in filas],
})

# Anchos de cada intervalo
tabla_intervalos['Ancho_IC'] = tabla_intervalos['IC_sup'] - tabla_intervalos['IC_inf']
tabla_intervalos['Ancho_IP'] = tabla_intervalos['IP_sup'] - tabla_intervalos['IP_inf']

tabla_intervalos

,Horas,Nota,y_hat,IC_inf,IC_sup,IP_inf,IP_sup,Ancho_IC,Ancho_IP
0,1.0000,3.2000,3.8255,3.0564,4.5945,2.3077,5.3432,1.5382,3.0356
1,2.0000,4.5000,4.5976,3.9453,5.2498,3.1355,6.0597,1.3045,2.9241
2,3.0000,5.0000,5.3697,4.8211,5.9183,3.9508,6.7885,1.0971,2.8377
3,4.0000,6.8000,6.1418,5.6750,6.6086,4.7525,7.5311,0.9336,2.7786
4,5.0000,7.0000,6.9139,6.4939,7.3340,5.5397,8.2882,0.8400,2.7485
5,6.0000,8.5000,7.6861,7.2660,8.1061,6.3118,9.0603,0.8400,2.7485
6,7.0000,9.0000,8.4582,7.9914,8.9250,7.0689,9.8475,0.9336,2.7786
7,8.0000,9.2000,9.2303,8.6817,9.7789,7.8115,10.6492,1.0971,2.8377
8,9.0000,9.8000,10.0024,9.3502,10.6547,8.5403,11.4645,1.3045,2.9241
9,10.0000,10.0000,10.7745,10.0055,11.5436,9.2568,12.2923,1.5382,3.0356


In [26]:
# Paso final :D!
# Celda extra para volver el notebook en un .html

!jupyter nbconvert --to html "Actividad2.ipynb"

[NbConvertApp] Converting notebook Actividad2.ipynb to html
[NbConvertApp] Writing 369091 bytes to Actividad2.html
